# Electronics Few-Shot PCF on Colab (Sampled + Sharded)

Runs Electronics few-shot PCF prediction using deterministic random sampling (seed=42) and disjoint shard runners for parallel Colab execution.

- Model: `Qwen/Qwen2.5-3B-Instruct`
- Prompt source: `src/carbon/retrieval.py` (`OpenAILLMClient.system_prompt` + `build_few_shot_prompt`)
- Terse mode + lower token cap for faster throughput
- Supports `test50` mode and full `sampled_shard` mode


## 1) Mount Google Drive

Mounts Drive so logs and predictions persist outside the Colab runtime.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2) Colab Setup and Repository Bootstrap

Clones the repo into Colab runtime, installs dependencies, and creates the Drive output folder.

In [ ]:
from pathlib import Path
import subprocess
import sys
import shutil

DRIVE_BASE = Path('/content/drive/MyDrive')
DRIVE_RUN_DIR = DRIVE_BASE / 'carbon-aware-recsys-colab' / 'pcf' / 'electronics_few_shot'
REPO_ROOT = Path('/content/carbon-aware-recsys')
REPO_URL = 'https://github.com/andersvestrum/carbon-aware-recsys.git'
REPO_BRANCH = 'reduce-pred-time-FS-el'

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)],
    check=True,
)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'accelerate'], check=True)

# Optional vLLM install for faster backend experiments.
vllm_installed = True
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True)
except Exception as exc:
    vllm_installed = False
    print(f'vLLM install failed (transformers backend still works): {exc}')

DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
print({'repo_root': str(REPO_ROOT), 'drive_run_dir': str(DRIVE_RUN_DIR), 'vllm_installed': vllm_installed})


## 3) Run Configuration and Logging

Sets model/category/run paths, output filenames, and logger handlers (console + Drive log file).

In [ ]:
import os
import sys
import time
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(REPO_ROOT))

from src.carbon.retrieval import (
    OpenAILLMClient,
    TransformersLLMClient,
    PCFRetrievalEstimator,
    RetrievalConfig,
    build_few_shot_prompt,
    parse_numeric_response,
    set_global_determinism,
)
from src.data.amazon_loader import (
    load_meta,
    load_amazon_electronics_data,
    load_amazon_hak_data,
    load_amazon_sao_data,
)

LLM_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
LLM_BACKEND = 'vllm'  # 'transformers' or 'vllm'
CATEGORY = 'electronics'
SEED = 42
TOP_K = 5
NUM_THREADS = 4
LLM_REASONING_STYLE = 'terse'
MAX_NEW_TOKENS = 96

# vLLM-only knobs
VLLM_TENSOR_PARALLEL_SIZE = 1
VLLM_GPU_MEMORY_UTILIZATION = 0.90

# Sample + shard controls
CATEGORY_TO_INTERACTION_LOADER = {
    'electronics': load_amazon_electronics_data,
    'home_and_kitchen': load_amazon_hak_data,
    'sports_and_outdoors': load_amazon_sao_data,
}

TARGET_SAMPLE_SIZE = 24_000
TARGET_NUM_SHARDS = 3
SAMPLE_SIZE_K = TARGET_SAMPLE_SIZE // 1000  # 24k fixed CF-eligible sample per category
SAMPLE_SIZE = SAMPLE_SIZE_K * 1000
SAMPLE_SEED = 42
NUM_SHARDS = TARGET_NUM_SHARDS
SHARD_ID = 0  # fixed shard id for this notebook file
RUN_MODE = 'sampled_shard'  # 'test50' or 'sampled_shard'
TEST_LIMIT = 50
RUN_LABEL = 'v2_cf_eligible'

if SAMPLE_SIZE != TARGET_SAMPLE_SIZE:
    raise ValueError(f'SAMPLE_SIZE must be {TARGET_SAMPLE_SIZE}, got {SAMPLE_SIZE}')
if NUM_SHARDS != TARGET_NUM_SHARDS:
    raise ValueError(f'NUM_SHARDS must be {TARGET_NUM_SHARDS}, got {NUM_SHARDS}')
if SHARD_ID not in {0, 1, 2}:
    raise ValueError(f'SHARD_ID must be 0, 1, or 2, got {SHARD_ID}')

DRIVE_RUN_BASE = DRIVE_BASE / 'carbon-aware-recsys-colab' / 'pcf' / 'electronics_sampled_shards_v2_cf_eligible'
RUN_TAG = f"xk{SAMPLE_SIZE_K}_seed{SAMPLE_SEED}_ns{NUM_SHARDS}_sh{SHARD_ID}_{LLM_BACKEND}_{RUN_LABEL}"
DRIVE_RUN_DIR = DRIVE_RUN_BASE / RUN_TAG
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PREDICTIONS_TEST = DRIVE_RUN_DIR / f'predictions_{RUN_TAG}_test50.csv'
OUTPUT_PREVIEW_TEST = DRIVE_RUN_DIR / f'predictions_{RUN_TAG}_test50_preview.csv'
OUTPUT_METADATA_TEST = DRIVE_RUN_DIR / f'run_metadata_{RUN_TAG}_test50.json'
OUTPUT_JSONL_TEST = DRIVE_RUN_DIR / f'predictions_{RUN_TAG}_test50.jsonl'

OUTPUT_PREDICTIONS_FULL = DRIVE_RUN_DIR / f'predictions_{RUN_TAG}_sampled_shard.csv'
OUTPUT_METADATA_FULL = DRIVE_RUN_DIR / f'run_metadata_{RUN_TAG}_sampled_shard.json'
OUTPUT_JSONL_FULL = DRIVE_RUN_DIR / f'predictions_{RUN_TAG}_sampled_shard.jsonl'

RUN_LOG = DRIVE_RUN_DIR / f'run_{RUN_TAG}.log'
LLM_CACHE = DRIVE_RUN_DIR / f'llm_cache_{RUN_TAG}.jsonl'

logger = logging.getLogger('electronics_few_shot_colab')
logger.setLevel(logging.INFO)
logger.handlers.clear()

fmt = logging.Formatter('%(asctime)s  %(levelname)s  %(message)s', datefmt='%H:%M:%S')
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(fmt)
file_handler = logging.FileHandler(RUN_LOG, mode='a', encoding='utf-8')
file_handler.setFormatter(fmt)

logger.addHandler(stream_handler)
logger.addHandler(file_handler)

logger.info('Configured run directory: %s', DRIVE_RUN_DIR)
logger.info('Run mode=%s | run_tag=%s', RUN_MODE, RUN_TAG)
logger.info('Model: %s | Backend: %s | Category: %s', LLM_MODEL, LLM_BACKEND, CATEGORY)
print({
    'run_mode': RUN_MODE,
    'run_tag': RUN_TAG,
    'llm_backend': LLM_BACKEND,
    'sample_size': SAMPLE_SIZE,
    'sample_seed': SAMPLE_SEED,
    'num_shards': NUM_SHARDS,
    'shard_id': SHARD_ID,
    'run_log': str(RUN_LOG),
    'llm_cache': str(LLM_CACHE),
})


## 4) Verify Canonical Prompt Source

Prints the exact system prompt and renders the few-shot template from `retrieval.py` for transparency.

In [ ]:
# Verify and display the exact prompt sources used from retrieval.py
print(f'System prompt for reasoning style={LLM_REASONING_STYLE}:')
print('-' * 80)
print(OpenAILLMClient.terse_system_prompt if LLM_REASONING_STYLE == 'terse' else OpenAILLMClient.system_prompt)
print('-' * 80)

sample_neighbors = [
    {'product_title': 'Stainless steel vacuum bottle 500ml', 'pcf': 4.1, 'similarity': 0.94, 'category': 'Kitchenware'},
    {'product_title': 'Insulated coffee tumbler 400ml', 'pcf': 2.8, 'similarity': 0.91, 'category': 'Kitchenware'},
    {'product_title': 'Aluminium sports bottle 600ml', 'pcf': 3.7, 'similarity': 0.88, 'category': 'Sporting goods'},
    {'product_title': 'Insulated food jar 300ml', 'pcf': 5.0, 'similarity': 0.85, 'category': 'Kitchenware'},
    {'product_title': 'Reusable travel mug 355ml', 'pcf': 1.4, 'similarity': 0.82, 'category': 'Kitchenware'},
]

print('\nRendered few-shot template from build_few_shot_prompt(...):')
print('-' * 80)
print(build_few_shot_prompt('Stainless steel double-wall travel mug 16 oz', sample_neighbors, reasoning_style=LLM_REASONING_STYLE))
print('-' * 80)


## 5) Initialize LLM Client and Retrieval Estimator

Loads the selected backend (`transformers` or `vllm`), sets deterministic config, and fits the Carbon Catalogue retrieval index.

In [ ]:
import torch
import transformers

transformers.logging.set_verbosity_error()

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

set_global_determinism(SEED, deterministic=True, num_threads=NUM_THREADS)


class VLLMNumericClient:
    system_prompt = OpenAILLMClient.system_prompt
    terse_system_prompt = OpenAILLMClient.terse_system_prompt

    def __init__(
        self,
        model: str,
        *,
        max_new_tokens: int = 128,
        reasoning_style: str = 'terse',
        tensor_parallel_size: int = 1,
        gpu_memory_utilization: float = 0.9,
    ):
        self.model = model
        self.max_new_tokens = max_new_tokens
        self.reasoning_style = reasoning_style
        self.system_prompt = (
            self.terse_system_prompt if reasoning_style == 'terse' else self.system_prompt
        )
        self.tensor_parallel_size = tensor_parallel_size
        self.gpu_memory_utilization = gpu_memory_utilization
        self._llm = None
        self._tokenizer = None

    @property
    def is_available(self):
        return True

    def _lazy_init(self):
        if self._llm is not None:
            return
        from vllm import LLM
        from transformers import AutoTokenizer

        self._llm = LLM(
            model=self.model,
            tensor_parallel_size=self.tensor_parallel_size,
            gpu_memory_utilization=self.gpu_memory_utilization,
            dtype='float16',
        )
        self._tokenizer = AutoTokenizer.from_pretrained(self.model)

    def predict_numeric(self, prompt: str) -> float:
        self._lazy_init()
        from vllm import SamplingParams

        messages = [
            {'role': 'system', 'content': self.system_prompt},
            {'role': 'user', 'content': prompt},
        ]
        prompt_text = self._tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        params = SamplingParams(temperature=0.0, max_tokens=self.max_new_tokens)
        out = self._llm.generate([prompt_text], params)
        text = out[0].outputs[0].text
        return parse_numeric_response(text)


if LLM_BACKEND == 'transformers':
    llm_client = TransformersLLMClient(
        model=LLM_MODEL,
        torch_dtype='float16',
        device_map='auto',
        max_new_tokens=MAX_NEW_TOKENS,
        reasoning_style=LLM_REASONING_STYLE,
    )
elif LLM_BACKEND == 'vllm':
    llm_client = VLLMNumericClient(
        model=LLM_MODEL,
        max_new_tokens=MAX_NEW_TOKENS,
        reasoning_style=LLM_REASONING_STYLE,
        tensor_parallel_size=VLLM_TENSOR_PARALLEL_SIZE,
        gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    )
else:
    raise ValueError("LLM_BACKEND must be 'transformers' or 'vllm'.")

config = RetrievalConfig(
    top_k=TOP_K,
    random_seed=SEED,
    deterministic=True,
    num_threads=NUM_THREADS,
    llm_workers=1,
    device='cpu',
)

estimator = PCFRetrievalEstimator(config)
logger.info('Fitting Carbon Catalogue retrieval index...')
estimator.fit_carbon_catalogue()
logger.info('Carbon Catalogue fit complete. Backend=%s', LLM_BACKEND)


## 6) Helper Functions for Structured Logs

Defines helpers to shape preview tables and write row-level JSONL prediction logs.

In [ ]:
def _prediction_log_columns(df: pd.DataFrame, top_k: int) -> list[str]:
    cols = [
        'parent_asin',
        'title',
        'main_category',
        'store',
        'price',
        'source_category',
        'few_shot_llm_pcf',
        'neighbor_average_pcf',
        'pcf',
        'pcf_source',
    ]
    for rank in range(1, top_k + 1):
        cols.extend([
            f'neighbor_{rank}_id',
            f'neighbor_{rank}_title',
            f'neighbor_{rank}_category',
            f'neighbor_{rank}_pcf',
            f'neighbor_{rank}_similarity',
        ])
    return [c for c in cols if c in df.columns]


def _write_prediction_jsonl(df: pd.DataFrame, output_path: Path, top_k: int) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cols = _prediction_log_columns(df, top_k)
    records = df[cols].to_dict(orient='records')
    with output_path.open('w', encoding='utf-8') as handle:
        for rec in records:
            handle.write(json.dumps(rec, ensure_ascii=True) + '\n')


def _preview_frame(df: pd.DataFrame) -> pd.DataFrame:
    preview_cols = [
        'parent_asin',
        'title',
        'few_shot_llm_pcf',
        'pcf',
        'pcf_source',
        'neighbor_1_title',
        'neighbor_1_similarity',
        'neighbor_2_title',
        'neighbor_2_similarity',
    ]
    preview_cols = [c for c in preview_cols if c in df.columns]
    return df[preview_cols].copy()


def _load_interaction_asins(category: str) -> set[str]:
    if category not in CATEGORY_TO_INTERACTION_LOADER:
        raise ValueError(f'Unsupported category: {category}')

    split_map = CATEGORY_TO_INTERACTION_LOADER[category](force_download=False)
    required_splits = ('train', 'val', 'test')
    missing_splits = [split for split in required_splits if split not in split_map]
    if missing_splits:
        raise ValueError(f'Missing interaction splits for {category}: {missing_splits}')

    asin_set: set[str] = set()
    for split in required_splits:
        split_df = split_map[split]
        if 'parent_asin' not in split_df.columns:
            raise ValueError(f"Split '{split}' for {category} is missing parent_asin column")
        asin_set.update(
            split_df['parent_asin']
            .dropna()
            .astype(str)
            .str.strip()
            .loc[lambda s: s.ne('')]
            .tolist()
        )
    return asin_set


def _build_sampled_shard_workload(
    meta_df: pd.DataFrame,
    *,
    category: str,
    sample_size: int,
    sample_seed: int,
    num_shards: int,
    shard_id: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    if sample_size != TARGET_SAMPLE_SIZE:
        raise ValueError(f'sample_size must be exactly {TARGET_SAMPLE_SIZE}, got {sample_size}')
    if num_shards != TARGET_NUM_SHARDS:
        raise ValueError(f'num_shards must be exactly {TARGET_NUM_SHARDS}, got {num_shards}')
    if shard_id < 0 or shard_id >= num_shards:
        raise ValueError(f'shard_id must be in [0, {num_shards - 1}]')

    if 'parent_asin' not in meta_df.columns:
        raise ValueError('meta_df must include parent_asin column')

    interaction_asins = _load_interaction_asins(category)
    meta_with_asin = meta_df.copy()
    meta_with_asin['parent_asin'] = meta_with_asin['parent_asin'].astype(str).str.strip()

    eligible_meta = (
        meta_with_asin.loc[meta_with_asin['parent_asin'].isin(interaction_asins)]
        .drop_duplicates(subset='parent_asin', keep='first')
        .sort_values('parent_asin', kind='stable')
        .reset_index(drop=True)
    )

    if len(eligible_meta) < sample_size:
        raise ValueError(
            f"Category '{category}' has {len(eligible_meta)} CF-eligible products, "
            f'but {sample_size} are required for this run.'
        )

    sampled = eligible_meta.sample(n=sample_size, random_state=sample_seed, replace=False).copy()
    sampled = sampled.sort_values('parent_asin', kind='stable').reset_index(drop=True)

    shard_splits = np.array_split(np.arange(len(sampled)), num_shards)
    shard_idx = shard_splits[shard_id]
    work_df = sampled.iloc[shard_idx].copy().reset_index(drop=True)

    if not work_df['parent_asin'].is_unique:
        raise ValueError('Shard workload contains duplicate parent_asin values')
    if not set(work_df['parent_asin']).issubset(interaction_asins):
        raise ValueError('Shard workload contains non-CF-eligible parent_asin values')

    summary = {
        'full_rows': int(len(meta_df)),
        'interaction_asin_rows': int(len(interaction_asins)),
        'eligible_rows': int(len(eligible_meta)),
        'sample_rows': int(len(sampled)),
        'sample_seed': int(sample_seed),
        'num_shards': int(num_shards),
        'shard_id': int(shard_id),
        'shard_rows': int(len(work_df)),
        'shard_sample_pos_start': int(shard_idx[0]) if len(shard_idx) else None,
        'shard_sample_pos_end_exclusive': int(shard_idx[-1] + 1) if len(shard_idx) else None,
        'sample_size_requested': int(sample_size),
        'sample_size_required': int(TARGET_SAMPLE_SIZE),
    }
    return sampled, work_df, shard_idx, summary


## 7) Build Deterministic Sample + Run Test50 Shard

Loads full Electronics metadata, builds deterministic sampled+sharded workload, then runs a 50-row test on the selected shard with timing and structured artifacts.


In [ ]:
# Build sampled-sharded workload and run test50 on this shard
logger.info('Loading full electronics metadata...')
meta_electronics = load_meta(CATEGORY)

sampled_df, work_df, shard_idx, shard_summary = _build_sampled_shard_workload(
    meta_electronics,
    category=CATEGORY,
    sample_size=SAMPLE_SIZE,
    sample_seed=SAMPLE_SEED,
    num_shards=NUM_SHARDS,
    shard_id=SHARD_ID,
)

logger.info(
    'CF eligibility summary | metadata rows: %d | interaction asins: %d | eligible rows: %d | sampled rows: %d | shard rows: %d',
    shard_summary['full_rows'],
    shard_summary['interaction_asin_rows'],
    shard_summary['eligible_rows'],
    len(sampled_df),
    len(work_df),
)
logger.info('Shard sample index range: [%s, %s)', shard_summary['shard_sample_pos_start'], shard_summary['shard_sample_pos_end_exclusive'])

if RUN_MODE != 'test50':
    logger.info("RUN_MODE=%s -> skipping test50 cell (set RUN_MODE='test50' to run this cell)", RUN_MODE)
else:
    test_df = work_df.head(TEST_LIMIT).copy()
    logger.info('Test rows selected from shard: %d', len(test_df))

    t0 = time.time()
    pred_test = estimator.predict_amazon_products(
        test_df,
        llm_client=llm_client,
        llm_model_name=LLM_MODEL,
        llm_cache_path=LLM_CACHE,
        llm_limit=None,
        llm_cache_only=False,
        llm_methods=('few_shot_llm',),
        llm_reasoning_style=LLM_REASONING_STYLE,
    )
    elapsed_test_min = (time.time() - t0) / 60
    logger.info('Test50 run complete in %.2f min', elapsed_test_min)

    pred_test.to_csv(OUTPUT_PREDICTIONS_TEST, index=False)
    preview_test = _preview_frame(pred_test)
    preview_test.to_csv(OUTPUT_PREVIEW_TEST, index=False)
    _write_prediction_jsonl(pred_test, OUTPUT_JSONL_TEST, TOP_K)

    metadata_test = {
        'run_type': 'test50_sharded',
        'run_mode': RUN_MODE,
        'run_tag': RUN_TAG,
        'category': CATEGORY,
        'llm_model': LLM_MODEL,
        'llm_backend': LLM_BACKEND,
        'llm_methods': ['few_shot_llm'],
        'llm_reasoning_style': LLM_REASONING_STYLE,
        'max_new_tokens': MAX_NEW_TOKENS,
        'rows': int(len(pred_test)),
        'seed': SEED,
        'top_k_neighbors': TOP_K,
        'elapsed_minutes': round(elapsed_test_min, 3),
        'sample_size_requested': int(SAMPLE_SIZE),
        'sample_size_required': int(TARGET_SAMPLE_SIZE),
        'interaction_asin_rows': int(shard_summary['interaction_asin_rows']),
        'eligible_rows': int(shard_summary['eligible_rows']),
        'sample_rows': int(len(sampled_df)),
        'sample_seed': int(SAMPLE_SEED),
        'num_shards': int(NUM_SHARDS),
        'num_shards_required': int(TARGET_NUM_SHARDS),
        'shard_id': int(SHARD_ID),
        'shard_rows': int(len(work_df)),
        'shard_sample_pos_start': shard_summary['shard_sample_pos_start'],
        'shard_sample_pos_end_exclusive': shard_summary['shard_sample_pos_end_exclusive'],
        'output_predictions_csv': str(OUTPUT_PREDICTIONS_TEST),
        'output_preview_csv': str(OUTPUT_PREVIEW_TEST),
        'output_jsonl': str(OUTPUT_JSONL_TEST),
        'run_log': str(RUN_LOG),
        'llm_cache': str(LLM_CACHE),
    }
    with OUTPUT_METADATA_TEST.open('w', encoding='utf-8') as handle:
        json.dump(metadata_test, handle, indent=2)

    print('Saved test artifacts:')
    print(' -', OUTPUT_PREDICTIONS_TEST)
    print(' -', OUTPUT_PREVIEW_TEST)
    print(' -', OUTPUT_JSONL_TEST)
    print(' -', OUTPUT_METADATA_TEST)

    display(preview_test.head(20))


## 8) Run Full Sampled Shard

Runs few-shot prediction on the entire sampled shard for this runner when `RUN_MODE='sampled_shard'`.


In [ ]:
# Run full sampled shard workload
if RUN_MODE != 'sampled_shard':
    logger.info("RUN_MODE=%s -> skipping full sampled shard cell (set RUN_MODE='sampled_shard' to run this)", RUN_MODE)
else:
    logger.info('Starting sampled shard run with %d rows...', len(work_df))
    t0 = time.time()
    pred_full = estimator.predict_amazon_products(
        work_df,
        llm_client=llm_client,
        llm_model_name=LLM_MODEL,
        llm_cache_path=LLM_CACHE,
        llm_limit=None,
        llm_cache_only=False,
        llm_methods=('few_shot_llm',),
        llm_reasoning_style=LLM_REASONING_STYLE,
    )
    elapsed_full_min = (time.time() - t0) / 60
    logger.info('Sampled shard run complete in %.2f min', elapsed_full_min)

    if not pred_full['parent_asin'].is_unique:
        raise ValueError('Predicted shard output contains duplicate parent_asin values')
    interaction_asins = _load_interaction_asins(CATEGORY)
    if not set(pred_full['parent_asin'].astype(str).str.strip()).issubset(interaction_asins):
        raise ValueError('Predicted shard output contains parent_asin outside CF interaction universe')

    pred_full.to_csv(OUTPUT_PREDICTIONS_FULL, index=False)
    _write_prediction_jsonl(pred_full, OUTPUT_JSONL_FULL, TOP_K)

    metadata_full = {
        'run_type': 'sampled_shard',
        'run_mode': RUN_MODE,
        'run_tag': RUN_TAG,
        'category': CATEGORY,
        'llm_model': LLM_MODEL,
        'llm_backend': LLM_BACKEND,
        'llm_methods': ['few_shot_llm'],
        'llm_reasoning_style': LLM_REASONING_STYLE,
        'max_new_tokens': MAX_NEW_TOKENS,
        'rows': int(len(pred_full)),
        'seed': SEED,
        'top_k_neighbors': TOP_K,
        'elapsed_minutes': round(elapsed_full_min, 3),
        'sample_size_requested': int(SAMPLE_SIZE),
        'sample_size_required': int(TARGET_SAMPLE_SIZE),
        'interaction_asin_rows': int(shard_summary['interaction_asin_rows']),
        'eligible_rows': int(shard_summary['eligible_rows']),
        'sample_rows': int(len(sampled_df)),
        'sample_seed': int(SAMPLE_SEED),
        'num_shards': int(NUM_SHARDS),
        'num_shards_required': int(TARGET_NUM_SHARDS),
        'shard_id': int(SHARD_ID),
        'shard_rows': int(len(work_df)),
        'output_unique_parent_asin_rows': int(pred_full['parent_asin'].nunique(dropna=True)),
        'output_parent_asin_in_interactions': True,
        'shard_sample_pos_start': shard_summary['shard_sample_pos_start'],
        'shard_sample_pos_end_exclusive': shard_summary['shard_sample_pos_end_exclusive'],
        'output_predictions_csv': str(OUTPUT_PREDICTIONS_FULL),
        'output_jsonl': str(OUTPUT_JSONL_FULL),
        'run_log': str(RUN_LOG),
        'llm_cache': str(LLM_CACHE),
    }
    with OUTPUT_METADATA_FULL.open('w', encoding='utf-8') as handle:
        json.dump(metadata_full, handle, indent=2)

    print('Saved sampled-shard artifacts:')
    print(' -', OUTPUT_PREDICTIONS_FULL)
    print(' -', OUTPUT_JSONL_FULL)
    print(' -', OUTPUT_METADATA_FULL)


## 9) Parallel Runner Quick Instructions\n
\n
1. Open one notebook per shard file (`*_00`, `*_01`, `*_02`), one Colab session each.\n
2. Keep `SAMPLE_SEED` and `LLM_BACKEND` consistent across sessions.\n
3. Do not override shard config in code; each notebook file has fixed `SHARD_ID`, `NUM_SHARDS=3`, and `SAMPLE_SIZE=24000`.\n
4. For timing smoke test, temporarily set `RUN_MODE='test50'` in each notebook and run all cells.\n
5. For production shard runs, use `RUN_MODE='sampled_shard'` and run all cells.\n
6. Merge shard outputs using `scripts/12_merge_sampled_shards.py`.\n

## 9) Output Locations and Live Logs

Shows exactly where outputs are written on mounted Drive and where to monitor live progress.

In [ ]:
# Where outputs are written while running
print('Drive output base:')
print(DRIVE_RUN_DIR)
print('\nRun details:')
print({
    'run_mode': RUN_MODE,
    'run_tag': RUN_TAG,
    'llm_backend': LLM_BACKEND,
    'sample_size_requested': SAMPLE_SIZE,
    'sample_seed': SAMPLE_SEED,
    'num_shards': NUM_SHARDS,
    'shard_id': SHARD_ID,
    'shard_rows_current_runner': len(work_df) if 'work_df' in globals() else None,
})

print('\nLive progress appears in:')
print(' - notebook cell output (progress bars + logs)')
print(' -', RUN_LOG)

print('\nCurrent files in output dir:')
for p in sorted(DRIVE_RUN_DIR.glob('*')):
    print(' -', p.name)

